In [3]:
#!/usr/bin/env python3
"""
QCM cross-validation / adjudication statistics
==============================================

Inputs
------
1) qcm__merged_not_yet_adjudicated.xlsx
   Expected to contain at least: CONSOLIDATION, ADJUDICATION_QUEUE,
   LONG_ANNOTATIONS (QUALITY_REPORT optional).

2) qcm_adjudicated.xlsx
   Same workbook after the senior adjudicator filled the adjudication columns
   in ADJUDICATION_QUEUE.

3) qcm_final.xlsx
   Final released QCM dataset. Only item_id and final_answer are mandatory;
   the script also audits any metadata/content columns shared with the
   consolidation workbook.

Outputs
-------
- qcm_validation_statistics.xlsx
- qcm_validation_report.md

Methodological distinction
--------------------------
For tacit items, validator-vs-validator agreement is reported separately from
three-way agreement involving the original pre-validation key. This avoids
presenting the original key as an independent human validator in the main
inter-rater reliability statistic.
"""

from __future__ import annotations

import math
import re
import unicodedata
from collections import Counter
from pathlib import Path
from statistics import NormalDist
from typing import Callable, Iterable, Sequence

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

VALID_ANSWERS = ("A", "B", "C", "D")
VALID_ANSWER_SET = set(VALID_ANSWERS)

PRE_SHEET = "CONSOLIDATION"
QUEUE_SHEET = "ADJUDICATION_QUEUE"
LONG_SHEET = "LONG_ANNOTATIONS"
QUALITY_SHEET = "QUALITY_REPORT"

ADJUDICATION_COLUMNS = [
    "adjudicated_by",
    "dispute_type",
    "final_answer",
    "status",
    "adjudication_comment",
]

STATUS_ALIASES = {
    "accepted": "accepted",
    "accept": "accepted",
    "accepte": "accepted",
    "acceptee": "accepted",
    "revised": "revised",
    "revise": "revised",
    "revisee": "revised",
    "rejected": "rejected",
    "reject": "rejected",
    "rejete": "rejected",
    "rejetee": "rejected",
}

ITEM_CONTENT_COLUMNS = [
    "item_type",
    "Axe",
    "Difficulty",
    "intent_group",
    "intent_category",
    "input",
    "response_A",
    "response_B",
    "response_C",
    "response_D",
]

BOOTSTRAP_SEED = 20260812


# -----------------------------------------------------------------------------
# Basic utilities
# -----------------------------------------------------------------------------


def clean_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_token(value: object) -> str:
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower().strip()
    text = re.sub(r"[_/–—-]+", " ", text)
    text = re.sub(r"[’'`]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_answer(value: object) -> str:
    text = normalize_token(value).upper()
    aliases = {
        "A": "A", "B": "B", "C": "C", "D": "D",
        "RESPONSE A": "A", "RESPONSE B": "B", "RESPONSE C": "C", "RESPONSE D": "D",
        "OPTION A": "A", "OPTION B": "B", "OPTION C": "C", "OPTION D": "D",
    }
    if text in aliases:
        return aliases[text]
    m = re.fullmatch(r"(?:RESPONSE|OPTION)?\s*([ABCD])", text)
    return m.group(1) if m else ""


def normalize_status(value: object) -> str:
    return STATUS_ALIASES.get(normalize_token(value), normalize_token(value))


def yes_no(value: bool) -> str:
    return "yes" if bool(value) else "no"


def safe_div(num: float, den: float) -> float:
    return float(num) / float(den) if den else float("nan")


def fmt_pct(x: object, digits: int = 1) -> str:
    try:
        x = float(x)
    except (TypeError, ValueError):
        return "NA"
    if math.isnan(x):
        return "NA"
    return f"{100*x:.{digits}f}%"


def fmt_num(x: object, digits: int = 3) -> str:
    try:
        x = float(x)
    except (TypeError, ValueError):
        return "NA"
    if math.isnan(x):
        return "NA"
    return f"{x:.{digits}f}"


def dedupe_join(values: Iterable[object], sep: str = " | ") -> str:
    out = []
    seen = set()
    for value in values:
        text = clean_text(value)
        if text and text not in seen:
            out.append(text)
            seen.add(text)
    return sep.join(out)


def require_columns(df: pd.DataFrame, columns: Sequence[str], label: str) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in {label}: {', '.join(missing)}")


def read_sheet(path: Path, sheet: str) -> pd.DataFrame:
    try:
        df = pd.read_excel(path, sheet_name=sheet, dtype=object)
    except ValueError as exc:
        raise ValueError(f"Sheet {sheet!r} not found in {path}") from exc
    df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed:")]
    df.columns = [str(c).strip() for c in df.columns]
    return df.dropna(how="all").reset_index(drop=True)


def read_first_sheet(path: Path) -> pd.DataFrame:
    xls = pd.ExcelFile(path)
    if not xls.sheet_names:
        raise ValueError(f"No sheet found in {path}")
    return read_sheet(path, xls.sheet_names[0])


def assert_unique_item_ids(df: pd.DataFrame, label: str) -> None:
    require_columns(df, ["item_id"], label)
    ids = df["item_id"].map(clean_text)
    if (ids == "").any():
        raise ValueError(f"Blank item_id found in {label}")
    dups = ids[ids.duplicated(keep=False)]
    if not dups.empty:
        raise ValueError(f"Duplicate item_id in {label}: {', '.join(dups.unique()[:20])}")


# -----------------------------------------------------------------------------
# Confidence intervals
# -----------------------------------------------------------------------------


def wilson_interval(successes: int, n: int, confidence: float = 0.95) -> tuple[float, float]:
    if n <= 0:
        return float("nan"), float("nan")
    z = NormalDist().inv_cdf(0.5 + confidence / 2)
    p = successes / n
    denom = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt((p * (1 - p) + z * z / (4 * n)) / n) / denom
    return max(0.0, center - half), min(1.0, center + half)


def bootstrap_ci_rows(
    data: pd.DataFrame,
    metric_fn: Callable[[pd.DataFrame], float],
    n_boot: int,
    rng: np.random.Generator,
    confidence: float = 0.95,
) -> tuple[float, float]:
    if n_boot <= 0 or len(data) < 2:
        return float("nan"), float("nan")
    values: list[float] = []
    n = len(data)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        sample = data.iloc[idx]
        value = metric_fn(sample)
        if value is not None and not math.isnan(float(value)):
            values.append(float(value))
    if len(values) < max(30, n_boot // 10):
        return float("nan"), float("nan")
    alpha = (1 - confidence) / 2
    return float(np.quantile(values, alpha)), float(np.quantile(values, 1 - alpha))


# -----------------------------------------------------------------------------
# Agreement statistics
# -----------------------------------------------------------------------------


def _valid_pair_df(df: pd.DataFrame, col_a: str, col_b: str) -> pd.DataFrame:
    mask = df[col_a].isin(VALID_ANSWER_SET) & df[col_b].isin(VALID_ANSWER_SET)
    return df.loc[mask, [col_a, col_b]].copy()


def raw_agreement(df: pd.DataFrame, col_a: str, col_b: str) -> float:
    d = _valid_pair_df(df, col_a, col_b)
    if d.empty:
        return float("nan")
    return float((d[col_a] == d[col_b]).mean())


def cohen_kappa(df: pd.DataFrame, col_a: str, col_b: str) -> float:
    d = _valid_pair_df(df, col_a, col_b)
    if d.empty:
        return float("nan")
    po = float((d[col_a] == d[col_b]).mean())
    pa = d[col_a].value_counts(normalize=True)
    pb = d[col_b].value_counts(normalize=True)
    pe = sum(float(pa.get(c, 0)) * float(pb.get(c, 0)) for c in VALID_ANSWERS)
    if math.isclose(1 - pe, 0.0):
        return float("nan")
    return (po - pe) / (1 - pe)


def gwet_ac1(df: pd.DataFrame, col_a: str, col_b: str) -> float:
    """Gwet's AC1 for two nominal raters."""
    d = _valid_pair_df(df, col_a, col_b)
    if d.empty:
        return float("nan")
    po = float((d[col_a] == d[col_b]).mean())
    q = len(VALID_ANSWERS)
    p = {}
    for c in VALID_ANSWERS:
        p1 = float((d[col_a] == c).mean())
        p2 = float((d[col_b] == c).mean())
        p[c] = (p1 + p2) / 2.0
    pe = sum(pk * (1 - pk) for pk in p.values()) / (q - 1)
    if math.isclose(1 - pe, 0.0):
        return float("nan")
    return (po - pe) / (1 - pe)


def krippendorff_alpha_nominal(matrix: Sequence[Sequence[object]]) -> float:
    """Krippendorff's alpha for nominal labels; missing/invalid values are ignored."""
    units: list[list[str]] = []
    pooled: list[str] = []
    for row in matrix:
        vals = [str(v) for v in row if str(v) in VALID_ANSWER_SET]
        units.append(vals)
        pooled.extend(vals)

    if len(pooled) < 2:
        return float("nan")

    # Observed disagreement: ordered pairs within each unit.
    obs_disagree = 0.0
    obs_pairs = 0.0
    for vals in units:
        m = len(vals)
        if m < 2:
            continue
        counts = Counter(vals)
        disagree = m * (m - 1) - sum(nc * (nc - 1) for nc in counts.values())
        obs_disagree += disagree
        obs_pairs += m * (m - 1)
    if obs_pairs == 0:
        return float("nan")
    do = obs_disagree / obs_pairs

    # Expected disagreement from pooled marginal distribution.
    n = len(pooled)
    counts = Counter(pooled)
    expected_disagree = n * (n - 1) - sum(nc * (nc - 1) for nc in counts.values())
    de = expected_disagree / (n * (n - 1))
    if math.isclose(de, 0.0):
        return float("nan")
    return 1 - do / de


def fleiss_kappa_complete(matrix: Sequence[Sequence[object]]) -> float:
    rows = [[str(v) for v in row] for row in matrix]
    rows = [row for row in rows if row and all(v in VALID_ANSWER_SET for v in row)]
    if not rows:
        return float("nan")
    n_raters = len(rows[0])
    rows = [row for row in rows if len(row) == n_raters]
    if n_raters < 2 or not rows:
        return float("nan")

    item_agreements = []
    totals = Counter()
    for row in rows:
        counts = Counter(row)
        totals.update(row)
        pi = (sum(v * v for v in counts.values()) - n_raters) / (n_raters * (n_raters - 1))
        item_agreements.append(pi)
    pbar = float(np.mean(item_agreements))
    total_ratings = len(rows) * n_raters
    pj = {c: totals.get(c, 0) / total_ratings for c in VALID_ANSWERS}
    pe = sum(v * v for v in pj.values())
    if math.isclose(1 - pe, 0.0):
        return float("nan")
    return (pbar - pe) / (1 - pe)


def mean_pairwise_agreement_3(df: pd.DataFrame, cols: Sequence[str]) -> float:
    if len(cols) != 3:
        raise ValueError("Exactly three columns required")
    d = df.loc[df[list(cols)].apply(lambda r: all(v in VALID_ANSWER_SET for v in r), axis=1), list(cols)]
    if d.empty:
        return float("nan")
    vals = []
    for a, b in [(cols[0], cols[1]), (cols[0], cols[2]), (cols[1], cols[2])]:
        vals.append(float((d[a] == d[b]).mean()))
    return float(np.mean(vals))


def n_valid_pairs(df: pd.DataFrame, a: str, b: str) -> int:
    return int((df[a].isin(VALID_ANSWER_SET) & df[b].isin(VALID_ANSWER_SET)).sum())


# -----------------------------------------------------------------------------
# Load + reconcile the three stages
# -----------------------------------------------------------------------------


def load_inputs(pre_path: Path, adjudicated_path: Path, final_path: Path):
    pre = read_sheet(pre_path, PRE_SHEET)
    pre_queue = read_sheet(pre_path, QUEUE_SHEET)
    adj_queue = read_sheet(adjudicated_path, QUEUE_SHEET)
    final = read_first_sheet(final_path)

    require_columns(
        pre,
        [
            "item_id", "item_type", "Axe", "Difficulty", "original_key",
            "annotator_1_id", "annotator_1_answer",
            "annotator_2_id", "annotator_2_answer",
            "automatic_decision", "provisional_answer",
        ],
        PRE_SHEET,
    )
    require_columns(adj_queue, ["item_id"] + ADJUDICATION_COLUMNS, f"{adjudicated_path.name}/{QUEUE_SHEET}")
    require_columns(final, ["item_id", "final_answer"], final_path.name)

    assert_unique_item_ids(pre, PRE_SHEET)
    assert_unique_item_ids(pre_queue, f"{pre_path.name}/{QUEUE_SHEET}")
    assert_unique_item_ids(adj_queue, f"{adjudicated_path.name}/{QUEUE_SHEET}")
    assert_unique_item_ids(final, final_path.name)

    # Normalize key fields.
    for df in (pre, pre_queue, adj_queue, final):
        df["item_id"] = df["item_id"].map(clean_text)
    pre["item_type"] = pre["item_type"].map(normalize_token)
    for c in ["original_key", "annotator_1_answer", "annotator_2_answer", "provisional_answer"]:
        pre[c] = pre[c].map(normalize_answer)
    adj_queue["final_answer"] = adj_queue["final_answer"].map(normalize_answer)
    adj_queue["status"] = adj_queue["status"].map(normalize_status)
    final["final_answer"] = final["final_answer"].map(normalize_answer)

    # Audit that adjudication only changed adjudication columns.
    common_nonadj = [c for c in pre_queue.columns if c in adj_queue.columns and c not in ADJUDICATION_COLUMNS]
    queue_compare = pre_queue.merge(adj_queue, on="item_id", how="outer", suffixes=("_pre", "_adj"), indicator=True)
    queue_audit_rows = []
    for _, row in queue_compare.iterrows():
        item_id = clean_text(row.get("item_id", ""))
        merge_state = clean_text(row.get("_merge", ""))
        if merge_state != "both":
            queue_audit_rows.append({
                "item_id": item_id,
                "issue": "queue_membership_changed",
                "column": "",
                "pre_value": merge_state,
                "adjudicated_value": merge_state,
            })
            continue
        for c in common_nonadj:
            if c == "item_id":
                continue
            pv = clean_text(row.get(f"{c}_pre", ""))
            av = clean_text(row.get(f"{c}_adj", ""))
            if pv != av:
                queue_audit_rows.append({
                    "item_id": item_id,
                    "issue": "non_adjudication_field_changed",
                    "column": c,
                    "pre_value": pv,
                    "adjudicated_value": av,
                })
    queue_audit = pd.DataFrame(queue_audit_rows)

    # Bring adjudicator decisions onto every initial item.
    adj_fields = adj_queue[["item_id"] + ADJUDICATION_COLUMNS].copy()
    resolved = pre.merge(adj_fields, on="item_id", how="left", suffixes=("", "_adjudicated"), validate="one_to_one")

    # For automatic accepts, the effective answer is the provisional answer.
    auto_accept = resolved["automatic_decision"].map(normalize_token).eq("accept")
    resolved["adjudication_status"] = resolved["status"].map(normalize_status)
    resolved["resolved_answer"] = ""
    resolved.loc[auto_accept, "resolved_answer"] = resolved.loc[auto_accept, "provisional_answer"]
    queued_with_answer = ~auto_accept & resolved["final_answer"].isin(VALID_ANSWER_SET)
    resolved.loc[queued_with_answer, "resolved_answer"] = resolved.loc[queued_with_answer, "final_answer"]

    # Final released answer, if retained.
    final_key = final[["item_id", "final_answer"]].rename(columns={"final_answer": "released_final_answer"})
    resolved = resolved.merge(final_key, on="item_id", how="left", validate="one_to_one")
    resolved["in_final"] = resolved["released_final_answer"].isin(VALID_ANSWER_SET)
    resolved["released_matches_resolved"] = (
        resolved["in_final"]
        & resolved["resolved_answer"].isin(VALID_ANSWER_SET)
        & resolved["released_final_answer"].eq(resolved["resolved_answer"])
    )

    return pre, pre_queue, adj_queue, final, resolved, queue_audit


# -----------------------------------------------------------------------------
# Metric builders
# -----------------------------------------------------------------------------


def metric_row(
    section: str,
    metric: str,
    value: object,
    n: object = "",
    ci_low: object = "",
    ci_high: object = "",
    note: str = "",
) -> dict[str, object]:
    return {
        "section": section,
        "metric": metric,
        "value": value,
        "n": n,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
        "note": note,
    }


def add_rate(rows: list[dict[str, object]], section: str, metric: str, successes: int, n: int, note: str = ""):
    value = safe_div(successes, n)
    lo, hi = wilson_interval(successes, n)
    rows.append(metric_row(section, metric, value, n, lo, hi, note))


def build_overall_metrics(resolved: pd.DataFrame, final: pd.DataFrame, adj_queue: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    n0 = len(resolved)
    n_final = len(final)
    documented_n = int(resolved["item_type"].eq("documented").sum())
    tacit_n = int(resolved["item_type"].eq("tacit").sum())
    auto_n = int(resolved["automatic_decision"].map(normalize_token).eq("accept").sum())
    queued_n = n0 - auto_n

    rows.extend([
        metric_row("dataset_flow", "initial_qcm_items", n0, n0),
        metric_row("dataset_flow", "documented_items", documented_n, documented_n),
        metric_row("dataset_flow", "tacit_items", tacit_n, tacit_n),
        metric_row("dataset_flow", "automatic_accept_items", auto_n, n0),
        metric_row("dataset_flow", "adjudication_queue_items", queued_n, n0),
        metric_row("dataset_flow", "released_final_items", n_final, n0),
    ])
    add_rate(rows, "dataset_flow", "automatic_accept_rate", auto_n, n0)
    add_rate(rows, "dataset_flow", "adjudication_rate", queued_n, n0)
    add_rate(rows, "dataset_flow", "final_retention_rate", n_final, n0)

    statuses = resolved.loc[~resolved["automatic_decision"].map(normalize_token).eq("accept"), "adjudication_status"]
    for status in ["accepted", "revised", "rejected"]:
        count = int(statuses.eq(status).sum())
        rows.append(metric_row("adjudication", f"status_{status}_count", count, queued_n))
        add_rate(rows, "adjudication", f"status_{status}_rate_among_queued", count, queued_n)

    completed = statuses.isin({"accepted", "revised", "rejected"})
    add_rate(rows, "adjudication", "adjudication_completion_rate", int(completed.sum()), queued_n)

    retained = resolved[resolved["in_final"]].copy()
    valid_orig_final = retained["original_key"].isin(VALID_ANSWER_SET) & retained["released_final_answer"].isin(VALID_ANSWER_SET)
    changed = valid_orig_final & retained["original_key"].ne(retained["released_final_answer"])
    add_rate(rows, "final_key", "original_key_changed_rate_among_released", int(changed.sum()), int(valid_orig_final.sum()))

    adjud_retained = retained.loc[~retained["automatic_decision"].map(normalize_token).eq("accept")].copy()
    valid_adj = adjud_retained["original_key"].isin(VALID_ANSWER_SET) & adjud_retained["released_final_answer"].isin(VALID_ANSWER_SET)
    changed_adj = valid_adj & adjud_retained["original_key"].ne(adjud_retained["released_final_answer"])
    add_rate(rows, "final_key", "original_key_changed_rate_among_adjudicated_released", int(changed_adj.sum()), int(valid_adj.sum()))

    # Whether final answer follows the provisional 2/3 majority when one exists.
    majority_cases = adjud_retained[
        adjud_retained["provisional_answer"].isin(VALID_ANSWER_SET)
        & adjud_retained["released_final_answer"].isin(VALID_ANSWER_SET)
    ]
    follows_majority = int(majority_cases["provisional_answer"].eq(majority_cases["released_final_answer"]).sum())
    add_rate(rows, "adjudication", "adjudicator_follows_provisional_majority_rate", follows_majority, len(majority_cases))

    # Key overturned specifically when both tacit validators agreed against it.
    tacit = adjud_retained[adjud_retained["item_type"].eq("tacit")].copy()
    two_validators_against_key = tacit[
        tacit["annotator_1_answer"].isin(VALID_ANSWER_SET)
        & tacit["annotator_2_answer"].isin(VALID_ANSWER_SET)
        & tacit["original_key"].isin(VALID_ANSWER_SET)
        & tacit["annotator_1_answer"].eq(tacit["annotator_2_answer"])
        & tacit["annotator_1_answer"].ne(tacit["original_key"])
        & tacit["released_final_answer"].isin(VALID_ANSWER_SET)
    ]
    follows_validators = int(
        two_validators_against_key["released_final_answer"].eq(two_validators_against_key["annotator_1_answer"]).sum()
    )
    add_rate(
        rows,
        "adjudication",
        "when_both_tacit_validators_disagree_with_key_final_follows_validators_rate",
        follows_validators,
        len(two_validators_against_key),
        "Diagnostic of how often senior adjudication overturns the initial key when both independent tacit validators agree against it.",
    )

    # Final workbook match against reconstructed answer.
    comparable = resolved[resolved["in_final"] & resolved["resolved_answer"].isin(VALID_ANSWER_SET)]
    matches = int(comparable["released_matches_resolved"].sum())
    add_rate(rows, "integrity", "released_answer_matches_reconstructed_resolution_rate", matches, len(comparable))

    return pd.DataFrame(rows)


def pair_metric_rows(
    section: str,
    label: str,
    df: pd.DataFrame,
    col_a: str,
    col_b: str,
    n_boot: int,
    rng: np.random.Generator,
) -> list[dict[str, object]]:
    d = _valid_pair_df(df, col_a, col_b)
    n = len(d)
    rows: list[dict[str, object]] = []
    if n == 0:
        for m in ["raw_agreement", "cohen_kappa", "gwet_ac1", "krippendorff_alpha_nominal"]:
            rows.append(metric_row(section, f"{label}__{m}", float("nan"), 0))
        return rows

    agreed = int((d[col_a] == d[col_b]).sum())
    raw = agreed / n
    lo, hi = wilson_interval(agreed, n)
    rows.append(metric_row(section, f"{label}__raw_agreement", raw, n, lo, hi))

    kappa = cohen_kappa(d, col_a, col_b)
    klo, khi = bootstrap_ci_rows(d, lambda x: cohen_kappa(x, col_a, col_b), n_boot, rng)
    rows.append(metric_row(section, f"{label}__cohen_kappa", kappa, n, klo, khi))

    ac1 = gwet_ac1(d, col_a, col_b)
    alo, ahi = bootstrap_ci_rows(d, lambda x: gwet_ac1(x, col_a, col_b), n_boot, rng)
    rows.append(metric_row(
        section,
        f"{label}__gwet_ac1",
        ac1,
        n,
        alo,
        ahi,
        "Useful alongside kappa when category prevalence is highly imbalanced.",
    ))

    alpha_fn = lambda x: krippendorff_alpha_nominal(x[[col_a, col_b]].values.tolist())
    alpha = alpha_fn(d)
    al_lo, al_hi = bootstrap_ci_rows(d, alpha_fn, n_boot, rng)
    rows.append(metric_row(section, f"{label}__krippendorff_alpha_nominal", alpha, n, al_lo, al_hi))
    return rows


def build_agreement_metrics(resolved: pd.DataFrame, long_annotations: pd.DataFrame | None, n_boot: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    rows: list[dict[str, object]] = []

    documented = resolved[resolved["item_type"].eq("documented")].copy()
    tacit = resolved[resolved["item_type"].eq("tacit")].copy()

    # Primary global IRR for the rotating pool of tacit validators: Krippendorff's alpha
    # computed on an item x actual-validator matrix. This is preferable to treating
    # assignment slots as if they were two fixed raters.
    if long_annotations is not None and not long_annotations.empty:
        la = long_annotations.copy()
        required = {"item_id", "item_type", "annotator_id", "annotator_answer"}
        if required.issubset(la.columns):
            la["item_id"] = la["item_id"].map(clean_text)
            la["item_type"] = la["item_type"].map(normalize_token)
            la["annotator_id"] = la["annotator_id"].map(clean_text)
            la["annotator_answer"] = la["annotator_answer"].map(normalize_answer)
            lat = la[
                la["item_type"].eq("tacit")
                & la["annotator_id"].ne("")
                & la["annotator_answer"].isin(VALID_ANSWER_SET)
            ].copy()
            if not lat.empty:
                pivot = lat.pivot_table(
                    index="item_id", columns="annotator_id", values="annotator_answer",
                    aggfunc="first", fill_value=""
                )
                if not pivot.empty:
                    alpha_pool_fn = lambda x: krippendorff_alpha_nominal(x.values.tolist())
                    alpha_pool = alpha_pool_fn(pivot)
                    ap_lo, ap_hi = bootstrap_ci_rows(pivot, alpha_pool_fn, n_boot, rng)
                    rows.append(metric_row(
                        "tacit_primary_inter_validator",
                        "krippendorff_alpha_actual_validator_pool",
                        alpha_pool,
                        len(pivot),
                        ap_lo,
                        ap_hi,
                        "Primary global IRR for tacit items because different pairs from the nine-validator pool rated different items.",
                    ))

    # Documented: initial construction key vs one independent validator.
    rows += pair_metric_rows(
        "documented",
        "initial_key_vs_validator",
        documented,
        "original_key",
        "annotator_1_answer",
        n_boot,
        rng,
    )

    # Tacit primary IRR: validator vs validator (the original key is NOT treated as a validator here).
    rows += pair_metric_rows(
        "tacit_primary_inter_validator",
        "validator1_vs_validator2",
        tacit,
        "annotator_1_answer",
        "annotator_2_answer",
        n_boot,
        rng,
    )

    # Tacit diagnostics against initial key.
    rows += pair_metric_rows(
        "tacit_key_diagnostic",
        "initial_key_vs_validator1",
        tacit,
        "original_key",
        "annotator_1_answer",
        n_boot,
        rng,
    )
    rows += pair_metric_rows(
        "tacit_key_diagnostic",
        "initial_key_vs_validator2",
        tacit,
        "original_key",
        "annotator_2_answer",
        n_boot,
        rng,
    )

    # Three-way operational consensus: initial key + two validators.
    cols3 = ["original_key", "annotator_1_answer", "annotator_2_answer"]
    complete = tacit.loc[tacit[cols3].apply(lambda r: all(v in VALID_ANSWER_SET for v in r), axis=1), cols3].copy()
    n3 = len(complete)
    if n3:
        unanimous = int(complete.apply(lambda r: len(set(r)) == 1, axis=1).sum())
        majority_2of3 = int(complete.apply(lambda r: sorted(Counter(r).values()) == [1, 2], axis=1).sum())
        all_different = int(complete.apply(lambda r: len(set(r)) == 3, axis=1).sum())
        for metric, successes in [
            ("three_way_unanimous_rate", unanimous),
            ("three_way_exactly_2_of_3_majority_rate", majority_2of3),
            ("three_way_all_different_rate", all_different),
        ]:
            lo, hi = wilson_interval(successes, n3)
            rows.append(metric_row("tacit_three_way_operational", metric, successes / n3, n3, lo, hi))

        fleiss_fn = lambda x: fleiss_kappa_complete(x[cols3].values.tolist())
        fleiss = fleiss_fn(complete)
        flo, fhi = bootstrap_ci_rows(complete, fleiss_fn, n_boot, rng)
        rows.append(metric_row(
            "tacit_three_way_operational",
            "fleiss_kappa_initial_key_plus_two_validators",
            fleiss,
            n3,
            flo,
            fhi,
            "Operational three-way consensus metric; do not describe the initial key as an independent validator.",
        ))

        alpha3_fn = lambda x: krippendorff_alpha_nominal(x[cols3].values.tolist())
        alpha3 = alpha3_fn(complete)
        a3lo, a3hi = bootstrap_ci_rows(complete, alpha3_fn, n_boot, rng)
        rows.append(metric_row(
            "tacit_three_way_operational",
            "krippendorff_alpha_initial_key_plus_two_validators",
            alpha3,
            n3,
            a3lo,
            a3hi,
            "Operational consensus statistic including the pre-validation key.",
        ))
        rows.append(metric_row(
            "tacit_three_way_operational",
            "mean_pairwise_raw_agreement_initial_key_plus_two_validators",
            mean_pairwise_agreement_3(complete, cols3),
            n3,
        ))

    # Post-hoc validator agreement with released final answer; descriptive, not independent reliability.
    released = resolved[resolved["in_final"]].copy()
    rows += pair_metric_rows(
        "posthoc_final_consistency",
        "validator1_vs_released_final",
        released,
        "annotator_1_answer",
        "released_final_answer",
        0,
        rng,
    )
    tacit_released = released[released["item_type"].eq("tacit")]
    rows += pair_metric_rows(
        "posthoc_final_consistency",
        "validator2_vs_released_final_tacit",
        tacit_released,
        "annotator_2_answer",
        "released_final_answer",
        0,
        rng,
    )

    overall = pd.DataFrame(rows)

    # Grouped point estimates by axis and difficulty (no bootstrap to keep runtime reasonable).
    grouped_rows: list[dict[str, object]] = []
    for group_col in ["Axe", "Difficulty"]:
        for group_value, g in resolved.groupby(group_col, dropna=False):
            group_value = clean_text(group_value) or "<blank>"
            doc = g[g["item_type"].eq("documented")]
            tac = g[g["item_type"].eq("tacit")]

            nd = n_valid_pairs(doc, "original_key", "annotator_1_answer")
            grouped_rows.append({
                "grouping": group_col,
                "group": group_value,
                "item_type": "documented",
                "metric": "raw_agreement_initial_key_vs_validator",
                "value": raw_agreement(doc, "original_key", "annotator_1_answer"),
                "n": nd,
            })
            grouped_rows.append({
                "grouping": group_col,
                "group": group_value,
                "item_type": "documented",
                "metric": "cohen_kappa_initial_key_vs_validator",
                "value": cohen_kappa(doc, "original_key", "annotator_1_answer"),
                "n": nd,
            })

            nt = n_valid_pairs(tac, "annotator_1_answer", "annotator_2_answer")
            grouped_rows.append({
                "grouping": group_col,
                "group": group_value,
                "item_type": "tacit",
                "metric": "raw_agreement_validator1_vs_validator2",
                "value": raw_agreement(tac, "annotator_1_answer", "annotator_2_answer"),
                "n": nt,
            })
            grouped_rows.append({
                "grouping": group_col,
                "group": group_value,
                "item_type": "tacit",
                "metric": "cohen_kappa_validator1_vs_validator2",
                "value": cohen_kappa(tac, "annotator_1_answer", "annotator_2_answer"),
                "n": nt,
            })
            grouped_rows.append({
                "grouping": group_col,
                "group": group_value,
                "item_type": "tacit",
                "metric": "gwet_ac1_validator1_vs_validator2",
                "value": gwet_ac1(tac, "annotator_1_answer", "annotator_2_answer"),
                "n": nt,
            })

            cols3 = ["original_key", "annotator_1_answer", "annotator_2_answer"]
            complete = tac.loc[tac[cols3].apply(lambda r: all(v in VALID_ANSWER_SET for v in r), axis=1), cols3]
            n3 = len(complete)
            unanimity = safe_div(int(complete.apply(lambda r: len(set(r)) == 1, axis=1).sum()), n3) if n3 else float("nan")
            grouped_rows.append({
                "grouping": group_col,
                "group": group_value,
                "item_type": "tacit",
                "metric": "three_way_unanimity_rate",
                "value": unanimity,
                "n": n3,
            })

    return overall, pd.DataFrame(grouped_rows)


def build_pair_specific_validator_metrics(long_annotations: pd.DataFrame | None) -> pd.DataFrame:
    """Agreement for each actual validator pair on tacit items."""
    if long_annotations is None or long_annotations.empty:
        return pd.DataFrame()
    required = {"item_id", "item_type", "annotator_id", "annotator_answer"}
    if not required.issubset(long_annotations.columns):
        return pd.DataFrame()

    la = long_annotations.copy()
    la["item_id"] = la["item_id"].map(clean_text)
    la["item_type"] = la["item_type"].map(normalize_token)
    la["annotator_id"] = la["annotator_id"].map(clean_text)
    la["annotator_answer"] = la["annotator_answer"].map(normalize_answer)
    la = la[
        la["item_type"].eq("tacit")
        & la["annotator_id"].ne("")
        & la["annotator_answer"].isin(VALID_ANSWER_SET)
    ]

    pair_rows = []
    for item_id, g in la.groupby("item_id"):
        g = g.drop_duplicates("annotator_id")
        if len(g) != 2:
            continue
        records = sorted(
            [(r.annotator_id, r.annotator_answer) for r in g.itertuples()],
            key=lambda x: x[0],
        )
        pair_rows.append({
            "item_id": item_id,
            "validator_a": records[0][0],
            "answer_a": records[0][1],
            "validator_b": records[1][0],
            "answer_b": records[1][1],
        })
    pairs = pd.DataFrame(pair_rows)
    if pairs.empty:
        return pairs

    out = []
    for (a, b), g in pairs.groupby(["validator_a", "validator_b"]):
        n = len(g)
        agreed = int(g["answer_a"].eq(g["answer_b"]).sum())
        lo, hi = wilson_interval(agreed, n)
        out.append({
            "validator_a": a,
            "validator_b": b,
            "n_shared_tacit_items": n,
            "raw_agreement": safe_div(agreed, n),
            "raw_agreement_ci95_low": lo,
            "raw_agreement_ci95_high": hi,
            "cohen_kappa": cohen_kappa(g, "answer_a", "answer_b"),
            "gwet_ac1": gwet_ac1(g, "answer_a", "answer_b"),
            "krippendorff_alpha_nominal": krippendorff_alpha_nominal(g[["answer_a", "answer_b"]].values.tolist()),
        })
    return pd.DataFrame(out).sort_values(["n_shared_tacit_items", "validator_a", "validator_b"], ascending=[False, True, True])


def build_distributions(resolved: pd.DataFrame) -> pd.DataFrame:
    rows = []

    def add_distribution(scope: str, item_type: str, series_name: str, series: pd.Series):
        valid = series[series.isin(VALID_ANSWER_SET)]
        n = len(valid)
        counts = valid.value_counts()
        for ans in VALID_ANSWERS:
            count = int(counts.get(ans, 0))
            rows.append({
                "scope": scope,
                "item_type": item_type,
                "distribution": series_name,
                "answer": ans,
                "count": count,
                "proportion": safe_div(count, n),
                "n_valid": n,
            })

    for item_type, g in [("ALL", resolved)] + list(resolved.groupby("item_type")):
        add_distribution("all_items", str(item_type), "initial_key", g["original_key"])
        add_distribution("all_items", str(item_type), "validator_1", g["annotator_1_answer"])
        add_distribution("all_items", str(item_type), "validator_2", g["annotator_2_answer"])
        add_distribution("released_items", str(item_type), "released_final", g["released_final_answer"])
    return pd.DataFrame(rows)


def build_confusion_matrices(resolved: pd.DataFrame) -> pd.DataFrame:
    specs = [
        ("documented_initial_key_vs_validator", resolved[resolved["item_type"].eq("documented")], "original_key", "annotator_1_answer"),
        ("tacit_validator1_vs_validator2", resolved[resolved["item_type"].eq("tacit")], "annotator_1_answer", "annotator_2_answer"),
        ("tacit_initial_key_vs_validator1", resolved[resolved["item_type"].eq("tacit")], "original_key", "annotator_1_answer"),
        ("tacit_initial_key_vs_validator2", resolved[resolved["item_type"].eq("tacit")], "original_key", "annotator_2_answer"),
    ]
    rows = []
    for label, df, a, b in specs:
        d = _valid_pair_df(df, a, b)
        n = len(d)
        for av in VALID_ANSWERS:
            row_den = int(d[a].eq(av).sum())
            for bv in VALID_ANSWERS:
                count = int((d[a].eq(av) & d[b].eq(bv)).sum())
                rows.append({
                    "comparison": label,
                    "row_variable": a,
                    "column_variable": b,
                    "row_answer": av,
                    "column_answer": bv,
                    "count": count,
                    "row_proportion": safe_div(count, row_den),
                    "overall_proportion": safe_div(count, n),
                    "n_valid_pairs": n,
                })
    return pd.DataFrame(rows)


def build_adjudication_breakdowns(resolved: pd.DataFrame) -> pd.DataFrame:
    rows = []
    queued_mask = ~resolved["automatic_decision"].map(normalize_token).eq("accept")

    def summarize_general(grouping: str, value: str, g_all: pd.DataFrame):
        total_n = len(g_all)
        g = g_all[~g_all["automatic_decision"].map(normalize_token).eq("accept")].copy()
        queued_n = len(g)
        status = g["adjudication_status"] if queued_n else pd.Series(dtype=object)
        released_valid = g["released_final_answer"].isin(VALID_ANSWER_SET) if queued_n else pd.Series(dtype=bool)
        changed = (
            released_valid
            & g["original_key"].isin(VALID_ANSWER_SET)
            & g["released_final_answer"].ne(g["original_key"])
        ) if queued_n else pd.Series(dtype=bool)
        rows.append({
            "grouping": grouping,
            "group": value,
            "total_items": total_n,
            "queued_items": queued_n,
            "adjudication_rate": safe_div(queued_n, total_n),
            "accepted": int(status.eq("accepted").sum()) if queued_n else 0,
            "revised": int(status.eq("revised").sum()) if queued_n else 0,
            "rejected": int(status.eq("rejected").sum()) if queued_n else 0,
            "released_items_among_queued": int(released_valid.sum()) if queued_n else 0,
            "key_changed_items_among_queued": int(changed.sum()) if queued_n else 0,
            "key_changed_rate_among_queued_released": safe_div(int(changed.sum()), int(released_valid.sum())) if queued_n else float("nan"),
        })

    summarize_general("ALL", "ALL", resolved)
    for col in ["item_type", "Axe", "Difficulty"]:
        if col not in resolved.columns:
            continue
        for value, g_all in resolved.groupby(col, dropna=False):
            summarize_general(col, clean_text(value) or "<blank>", g_all)

    # Queue-only descriptive distributions for why cases were escalated and how they were adjudicated.
    queued = resolved.loc[queued_mask].copy()
    for col in ["automatic_decision", "decision_reason", "dispute_type", "adjudication_status"]:
        if col not in queued.columns:
            continue
        counts = queued[col].map(clean_text).replace("", "<blank>").value_counts(dropna=False)
        for value, count in counts.items():
            rows.append({
                "grouping": f"queued_{col}",
                "group": value,
                "total_items": len(queued),
                "queued_items": int(count),
                "adjudication_rate": safe_div(int(count), len(queued)),
                "accepted": "",
                "revised": "",
                "rejected": "",
                "released_items_among_queued": "",
                "key_changed_items_among_queued": "",
                "key_changed_rate_among_queued_released": "",
            })
    return pd.DataFrame(rows)


def build_validator_summary(resolved: pd.DataFrame, long_annotations: pd.DataFrame | None) -> pd.DataFrame:
    if long_annotations is None or long_annotations.empty:
        # Fallback from the wide consolidation table.
        pieces = []
        for slot in [1, 2]:
            id_col = f"annotator_{slot}_id"
            ans_col = f"annotator_{slot}_answer"
            if id_col not in resolved.columns:
                continue
            p = resolved[["item_id", "item_type", "original_key", "released_final_answer", id_col, ans_col]].copy()
            p = p.rename(columns={id_col: "annotator_id", ans_col: "annotator_answer"})
            p = p[p["annotator_id"].map(clean_text).ne("")]
            pieces.append(p)
        long = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()
    else:
        long = long_annotations.copy()
        long["item_id"] = long["item_id"].map(clean_text)
        long["annotator_id"] = long["annotator_id"].map(clean_text)
        long["annotator_answer"] = long["annotator_answer"].map(normalize_answer)
        keep_res = resolved[["item_id", "original_key", "released_final_answer"]].copy()
        long = long.merge(keep_res, on="item_id", how="left", validate="many_to_one")

    if long.empty:
        return pd.DataFrame()

    rows = []
    for ann, g in long.groupby("annotator_id"):
        valid_key = g["annotator_answer"].isin(VALID_ANSWER_SET) & g["original_key"].isin(VALID_ANSWER_SET)
        valid_final = g["annotator_answer"].isin(VALID_ANSWER_SET) & g["released_final_answer"].isin(VALID_ANSWER_SET)
        row = {
            "annotator_id": ann,
            "n_annotations": len(g),
            "documented_n": int(g["item_type"].map(normalize_token).eq("documented").sum()),
            "tacit_n": int(g["item_type"].map(normalize_token).eq("tacit").sum()),
            "agreement_with_initial_key": safe_div(int((g.loc[valid_key, "annotator_answer"] == g.loc[valid_key, "original_key"]).sum()), int(valid_key.sum())),
            "n_for_initial_key_agreement": int(valid_key.sum()),
            "agreement_with_released_final": safe_div(int((g.loc[valid_final, "annotator_answer"] == g.loc[valid_final, "released_final_answer"]).sum()), int(valid_final.sum())),
            "n_for_final_agreement": int(valid_final.sum()),
        }
        if "annotation_complete" in g.columns:
            complete = g["annotation_complete"].map(normalize_token).isin({"true", "yes", "1"})
            if g["annotation_complete"].dtype == bool:
                complete = g["annotation_complete"]
            row["completion_rate"] = safe_div(int(complete.sum()), len(g))
        if "flag_language" in g.columns:
            row["language_flag_rate"] = safe_div(int(g["flag_language"].map(normalize_token).eq("probleme").sum()), len(g))
        if "flag_safety" in g.columns:
            row["safety_flag_rate"] = safe_div(int(g["flag_safety"].map(normalize_token).eq("probleme").sum()), len(g))
        if "flag_item" in g.columns:
            valid_flag = g["flag_item"].map(normalize_token)
            row["item_flag_rate"] = safe_div(int((valid_flag.ne("") & valid_flag.ne("ok")).sum()), len(g))
        if "annotator_sources" in g.columns:
            doc = g[g["item_type"].map(normalize_token).eq("documented")]
            row["documented_source_completion_rate"] = safe_div(int(doc["annotator_sources"].map(clean_text).ne("").sum()), len(doc))
        rows.append(row)
    return pd.DataFrame(rows).sort_values("annotator_id")


def build_final_integrity(pre: pd.DataFrame, resolved: pd.DataFrame, final: pd.DataFrame, queue_audit: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows: list[dict[str, object]] = []
    issues: list[dict[str, object]] = []

    pre_ids = set(pre["item_id"])
    final_ids = set(final["item_id"])
    missing_from_final = sorted(pre_ids - final_ids)
    extra_in_final = sorted(final_ids - pre_ids)
    invalid_final = final.loc[~final["final_answer"].isin(VALID_ANSWER_SET), "item_id"].tolist()

    rows.extend([
        {"check": "initial_ids", "value": len(pre_ids), "pass": ""},
        {"check": "final_ids", "value": len(final_ids), "pass": ""},
        {"check": "missing_from_final", "value": len(missing_from_final), "pass": ""},
        {"check": "extra_in_final", "value": len(extra_in_final), "pass": yes_no(len(extra_in_final) == 0)},
        {"check": "invalid_final_answers", "value": len(invalid_final), "pass": yes_no(len(invalid_final) == 0)},
        {"check": "unexpected_changes_during_adjudication", "value": len(queue_audit), "pass": yes_no(queue_audit.empty)},
    ])

    for item_id in missing_from_final:
        r = resolved.loc[resolved["item_id"].eq(item_id)]
        status = clean_text(r.iloc[0]["adjudication_status"]) if not r.empty else ""
        issues.append({"item_id": item_id, "issue": "missing_from_final", "details": f"adjudication_status={status}"})
    for item_id in extra_in_final:
        issues.append({"item_id": item_id, "issue": "extra_in_final", "details": "item_id not present in pre-adjudication consolidation"})
    for item_id in invalid_final:
        issues.append({"item_id": item_id, "issue": "invalid_final_answer", "details": "final_answer is not A/B/C/D"})

    # Missing from final is expected only for rejected items.
    unexpected_missing = 0
    for item_id in missing_from_final:
        r = resolved.loc[resolved["item_id"].eq(item_id)]
        status = clean_text(r.iloc[0]["adjudication_status"]) if not r.empty else ""
        if status != "rejected":
            unexpected_missing += 1
    rows.append({"check": "missing_from_final_not_marked_rejected", "value": unexpected_missing, "pass": yes_no(unexpected_missing == 0)})

    # Released answer should equal reconstructed resolved answer for retained items.
    comparable = resolved[resolved["in_final"] & resolved["resolved_answer"].isin(VALID_ANSWER_SET)]
    mism = comparable[~comparable["released_matches_resolved"]]
    rows.append({"check": "released_answer_resolution_mismatches", "value": len(mism), "pass": yes_no(mism.empty)})
    for r in mism.itertuples():
        issues.append({
            "item_id": r.item_id,
            "issue": "released_answer_resolution_mismatch",
            "details": f"resolved={r.resolved_answer}; qcm_final={r.released_final_answer}",
        })

    # Content consistency for columns available in both files.
    common_content = [c for c in ITEM_CONTENT_COLUMNS if c in pre.columns and c in final.columns]
    if common_content:
        left = pre[["item_id"] + common_content].copy()
        right = final[["item_id"] + common_content].copy()
        cmp = left.merge(right, on="item_id", how="inner", suffixes=("_pre", "_final"))
        content_mismatches = 0
        for r in cmp.itertuples(index=False):
            for c in common_content:
                pv = clean_text(getattr(r, f"{c}_pre"))
                fv = clean_text(getattr(r, f"{c}_final"))
                if pv != fv:
                    content_mismatches += 1
                    issues.append({
                        "item_id": getattr(r, "item_id"),
                        "issue": "final_content_changed",
                        "details": f"column={c}; pre={pv!r}; final={fv!r}",
                    })
        rows.append({"check": "final_content_cell_mismatches", "value": content_mismatches, "pass": yes_no(content_mismatches == 0)})

    return pd.DataFrame(rows), pd.DataFrame(issues)


def build_item_audit(resolved: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "item_id", "item_type", "Axe", "Difficulty", "intent_group", "intent_category",
        "original_key", "annotator_1_id", "annotator_1_answer", "annotator_2_id", "annotator_2_answer",
        "vote_distribution", "provisional_answer", "automatic_decision", "decision_reason",
        "mandatory_flag_review", "language_review_recommended",
        "adjudicated_by", "dispute_type", "adjudication_status", "adjudication_comment",
        "resolved_answer", "released_final_answer", "in_final", "released_matches_resolved",
    ]
    cols = [c for c in cols if c in resolved.columns]
    out = resolved[cols].copy()
    out["original_key_changed_in_release"] = (
        out["released_final_answer"].isin(VALID_ANSWER_SET)
        & out["original_key"].isin(VALID_ANSWER_SET)
        & out["released_final_answer"].ne(out["original_key"])
    )
    return out


# -----------------------------------------------------------------------------
# Paper-ready markdown report
# -----------------------------------------------------------------------------


def lookup_metric(metrics: pd.DataFrame, metric: str) -> pd.Series | None:
    m = metrics.loc[metrics["metric"].eq(metric)]
    return m.iloc[0] if not m.empty else None


def build_markdown_report(
    overall: pd.DataFrame,
    agreement: pd.DataFrame,
    final_integrity: pd.DataFrame,
    pre_name: str,
    adj_name: str,
    final_name: str,
) -> str:
    def val(df: pd.DataFrame, metric: str, pct: bool = False) -> str:
        r = lookup_metric(df, metric)
        if r is None:
            return "NA"
        return fmt_pct(r["value"]) if pct else fmt_num(r["value"])

    def count(metric: str) -> str:
        r = lookup_metric(overall, metric)
        return str(int(r["value"])) if r is not None and not pd.isna(r["value"]) else "NA"

    lines = [
        "# QCM cross-validation and adjudication statistics",
        "",
        f"Inputs: `{pre_name}`, `{adj_name}`, `{final_name}`.",
        "",
        "## Dataset flow",
        "",
        f"- Initial QCM items: **{count('initial_qcm_items')}**.",
        f"- Documented items: **{count('documented_items')}**; tacit items: **{count('tacit_items')}**.",
        f"- Automatic acceptance rate: **{val(overall, 'automatic_accept_rate', True)}**.",
        f"- Adjudication rate: **{val(overall, 'adjudication_rate', True)}**.",
        f"- Final retention rate: **{val(overall, 'final_retention_rate', True)}**.",
        f"- Initial-key change rate among released items: **{val(overall, 'original_key_changed_rate_among_released', True)}**.",
        "",
        "## Reliability statistics",
        "",
        f"- Documented items — initial key vs independent validator: raw agreement **{val(agreement, 'initial_key_vs_validator__raw_agreement', True)}**, Cohen's κ **{val(agreement, 'initial_key_vs_validator__cohen_kappa')}**, Krippendorff's α **{val(agreement, 'initial_key_vs_validator__krippendorff_alpha_nominal')}**, Gwet's AC1 **{val(agreement, 'initial_key_vs_validator__gwet_ac1')}**.",
        f"- Tacit items — rotating validator pool (primary global inter-rater reliability): Krippendorff's nominal α **{val(agreement, 'krippendorff_alpha_actual_validator_pool')}**. The direct pairwise answer agreement is **{val(agreement, 'validator1_vs_validator2__raw_agreement', True)}**; pooled slot-wise Cohen's κ **{val(agreement, 'validator1_vs_validator2__cohen_kappa')}** and Gwet's AC1 **{val(agreement, 'validator1_vs_validator2__gwet_ac1')}** are reported as supplementary diagnostics.",
        f"- Tacit items — three-way unanimity (initial key + two validators): **{val(agreement, 'three_way_unanimous_rate', True)}**; Fleiss' κ **{val(agreement, 'fleiss_kappa_initial_key_plus_two_validators')}**.",
        "",
        "**Interpretation note.** Because different pairs from the nine-person validator pool reviewed different tacit items, Krippendorff's nominal α computed over the actual item-by-validator matrix is the primary global IRR statistic. Pooled slot-wise κ/AC1 are supplementary. Three-way statistics that also include the original pre-validation key are operational consensus measures, not reliability among three independent validators.",
        "",
        "## Recommended paper reporting",
        "",
        "Report raw agreement together with chance-corrected coefficients and their 95% confidence intervals. For tacit items with rotating validator pairs, emphasize Krippendorff's nominal α across the actual validator pool and raw pairwise agreement. Report pooled Cohen's κ and Gwet's AC1 as complementary diagnostics, and use the three-way unanimity/majority statistics to describe the adjudication protocol.",
        "",
        "## Integrity checks",
        "",
    ]
    for _, r in final_integrity.iterrows():
        pass_value = clean_text(r.get("pass", ""))
        suffix = f" (pass={pass_value})" if pass_value else ""
        lines.append(f"- {r.get('check', '')}: {r.get('value', '')}{suffix}")
    return "\n".join(lines) + "\n"


# -----------------------------------------------------------------------------
# Excel export
# -----------------------------------------------------------------------------


def autosize_worksheet(ws, max_width: int = 70) -> None:
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter

    header_fill = PatternFill("solid", fgColor="D9EAF7")
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False

    for col_idx, col_cells in enumerate(ws.iter_cols(1, ws.max_column), start=1):
        max_len = 0
        for cell in col_cells[: min(ws.max_row, 300)]:
            value = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, min(len(value), max_width))
            cell.alignment = Alignment(vertical="top", wrap_text=True)
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 10), max_width)


def write_excel(
    output: Path,
    overall: pd.DataFrame,
    agreement: pd.DataFrame,
    grouped_agreement: pd.DataFrame,
    pair_specific: pd.DataFrame,
    distributions: pd.DataFrame,
    confusion_matrices: pd.DataFrame,
    adjudication: pd.DataFrame,
    validator_summary: pd.DataFrame,
    final_integrity: pd.DataFrame,
    integrity_issues: pd.DataFrame,
    queue_audit: pd.DataFrame,
    quality_report: pd.DataFrame | None,
    item_audit: pd.DataFrame,
) -> None:
    readme = pd.DataFrame({
        "section": [
            "Purpose",
            "Primary IRR",
            "Documented items",
            "Tacit items",
            "Confidence intervals",
            "Cohen kappa",
            "Gwet AC1",
            "Krippendorff alpha",
            "Fleiss kappa",
            "Important caveat",
        ],
        "description": [
            "Statistics for the QCM cross-validation, adjudication and final-release pipeline.",
            "For tacit items, Krippendorff's nominal alpha across the actual rotating validator pool is the primary global IRR statistic; raw pairwise agreement is also central.",
            "Initial key vs one blinded independent validator; source verification was required by the validation protocol.",
            "Two independent validators; the original pre-validation key is separately included in operational three-way consensus metrics.",
            "Rates use Wilson 95% CIs. Overall chance-corrected agreement coefficients use item-level bootstrap percentile 95% CIs.",
            "Conventional chance-corrected agreement coefficient; can be affected by prevalence/marginal imbalance.",
            "Robust complementary agreement coefficient, useful when kappa is depressed by prevalence imbalance.",
            "Nominal-scale reliability coefficient; implemented directly and tolerant of missing ratings in principle.",
            "Reported for the operational three-way set (initial key + two validators), not as three independent validators.",
            "Do not call the initial/pre-validation key a gold label before adjudication. Treat it as a construction-stage proposal that can be overturned.",
        ],
    })

    with pd.ExcelWriter(output, engine="openpyxl") as writer:
        readme.to_excel(writer, index=False, sheet_name="README")
        overall.to_excel(writer, index=False, sheet_name="OVERALL_METRICS")
        agreement.to_excel(writer, index=False, sheet_name="AGREEMENT")
        grouped_agreement.to_excel(writer, index=False, sheet_name="AGREEMENT_BY_GROUP")
        pair_specific.to_excel(writer, index=False, sheet_name="VALIDATOR_PAIRS")
        distributions.to_excel(writer, index=False, sheet_name="ANSWER_DISTRIBUTIONS")
        confusion_matrices.to_excel(writer, index=False, sheet_name="CONFUSION_MATRICES")
        adjudication.to_excel(writer, index=False, sheet_name="ADJUDICATION")
        validator_summary.to_excel(writer, index=False, sheet_name="VALIDATORS")
        final_integrity.to_excel(writer, index=False, sheet_name="FINAL_INTEGRITY")
        integrity_issues.to_excel(writer, index=False, sheet_name="INTEGRITY_ISSUES")
        queue_audit.to_excel(writer, index=False, sheet_name="QUEUE_CHANGE_AUDIT")
        if quality_report is not None:
            quality_report.to_excel(writer, index=False, sheet_name="INPUT_QUALITY_REPORT")
        item_audit.to_excel(writer, index=False, sheet_name="ITEM_LEVEL_AUDIT")

    from openpyxl import load_workbook
    wb = load_workbook(output)
    for ws in wb.worksheets:
        autosize_worksheet(ws)
    wb.save(output)


# -----------------------------------------------------------------------------
# Main Execution (Jupyter Notebook Ready)
# -----------------------------------------------------------------------------


def main(
    pre: Path = Path("qcm__merged_not_yet_adjudicated.xlsx"),
    adjudicated: Path = Path("qcm_adjudicated.xlsx"),
    final: Path = Path("qcm_final.xlsx"),
    output: Path = Path("qcm_validation_statistics.xlsx"),
    report: Path = Path("qcm_validation_report.md"),
    bootstrap: int = 2000,
) -> None:
    for path in [pre, adjudicated, final]:
        if not path.exists():
            raise FileNotFoundError(f"Input file not found: {path}")

    pre_df, pre_queue, adj_queue, final_df, resolved, queue_audit = load_inputs(
        pre, adjudicated, final
    )

    # Optional detailed inputs.
    long_annotations = None
    quality_report = None
    try:
        long_annotations = read_sheet(pre, LONG_SHEET)
    except ValueError:
        pass
    try:
        quality_report = read_sheet(pre, QUALITY_SHEET)
    except ValueError:
        pass

    overall = build_overall_metrics(resolved, final_df, adj_queue)
    agreement, grouped_agreement = build_agreement_metrics(resolved, long_annotations, max(0, bootstrap))
    pair_specific = build_pair_specific_validator_metrics(long_annotations)
    distributions = build_distributions(resolved)
    confusion_matrices = build_confusion_matrices(resolved)
    adjudication = build_adjudication_breakdowns(resolved)
    validator_summary = build_validator_summary(resolved, long_annotations)
    final_integrity, integrity_issues = build_final_integrity(pre_df, resolved, final_df, queue_audit)
    item_audit = build_item_audit(resolved)

    output.parent.mkdir(parents=True, exist_ok=True)
    report.parent.mkdir(parents=True, exist_ok=True)

    write_excel(
        output,
        overall,
        agreement,
        grouped_agreement,
        pair_specific,
        distributions,
        confusion_matrices,
        adjudication,
        validator_summary,
        final_integrity,
        integrity_issues,
        queue_audit,
        quality_report,
        item_audit,
    )

    report_content = build_markdown_report(
        overall,
        agreement,
        final_integrity,
        pre.name,
        adjudicated.name,
        final.name,
    )
    report.write_text(report_content, encoding="utf-8")

    print("=" * 80)
    print("QCM validation statistics complete")
    print(f"Pre-adjudication: {pre}")
    print(f"Adjudicated:      {adjudicated}")
    print(f"Final:            {final}")
    print(f"Statistics:       {output}")
    print(f"Markdown report:  {report}")
    print(f"Bootstrap runs:   {bootstrap}")
    print("=" * 80)

    # Compact headline metrics in console.
    headline = [
        "adjudication_rate",
        "final_retention_rate",
        "original_key_changed_rate_among_released",
    ]
    for name in headline:
        r = lookup_metric(overall, name)
        if r is not None:
            print(f"{name}: {fmt_pct(r['value'])}")

    for name in [
        "initial_key_vs_validator__cohen_kappa",
        "krippendorff_alpha_actual_validator_pool",
        "validator1_vs_validator2__cohen_kappa",
        "validator1_vs_validator2__gwet_ac1",
        "three_way_unanimous_rate",
    ]:
        r = lookup_metric(agreement, name)
        if r is not None:
            if "rate" in name:
                print(f"{name}: {fmt_pct(r['value'])}")
            else:
                print(f"{name}: {fmt_num(r['value'])}")


# Execute directly when running cell
main()

QCM validation statistics complete
Pre-adjudication: qcm__merged_not_yet_adjudicated.xlsx
Adjudicated:      qcm_adjudicated.xlsx
Final:            qcm_final.xlsx
Statistics:       qcm_validation_statistics.xlsx
Markdown report:  qcm_validation_report.md
Bootstrap runs:   2000
adjudication_rate: 19.6%
final_retention_rate: 98.1%
original_key_changed_rate_among_released: 4.7%
initial_key_vs_validator__cohen_kappa: 0.881
krippendorff_alpha_actual_validator_pool: 0.928
validator1_vs_validator2__cohen_kappa: 0.928
validator1_vs_validator2__gwet_ac1: 0.930
three_way_unanimous_rate: 93.1%
